[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NU-MSE-LECTURES/465-WINTER2026/blob/main/Week_04/assignments/assignment_04_combined.ipynb)

# Assignment 04: Neural Network for Atomic-Level Pattern Recognition

## Overview
This assignment combines supervised learning, deep learning, and visualization techniques to develop a neural network for atomic-level pattern recognition in microscopy images.

## Learning Objectives
- Apply CNNs to microscopy-relevant pattern recognition tasks
- Generate synthetic microscopy-inspired datasets
- Train and evaluate deep learning models
- Create comprehensive visualization summaries
- Save and document model results

## Assignment Tasks
1. Generate synthetic atomic pattern dataset
2. Build and train a CNN classifier
3. Evaluate model performance
4. **Create and save a four-panel summary figure**

## Setup and Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

try:
    import tensorflow
    print("TensorFlow already installed")
except ImportError:
    print("Installing TensorFlow...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    print("TensorFlow installed successfully")

# Colab setup
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab.")
except ImportError:
    IN_COLAB = False
    print("Running locally.")

In [ ]:
# Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# Image processing
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print("All libraries imported successfully!")

## Task 1: Generate Synthetic Atomic Pattern Dataset

Create a dataset of synthetic atomic patterns representing different crystal structures or defects.

In [ ]:
def generate_atomic_patterns(n_samples=300, img_size=64, n_classes=4):
    """
    Generate synthetic atomic patterns for different crystal structures.
    
    Args:
        n_samples: Total number of samples to generate
        img_size: Size of square images
        n_classes: Number of pattern classes (e.g., different crystal structures)
    
    Returns:
        images: Array of shape (n_samples, img_size, img_size, 1)
        labels: Array of class labels (n_samples,)
    """
    images = []
    labels = []
    
    samples_per_class = n_samples // n_classes
    
    for class_id in range(n_classes):
        for _ in range(samples_per_class):
            img = np.zeros((img_size, img_size))
            
            if class_id == 0:  # Square lattice
                spacing = 8
                for i in range(spacing//2, img_size, spacing):
                    for j in range(spacing//2, img_size, spacing):
                        img[i:i+2, j:j+2] = 1.0
            
            elif class_id == 1:  # Hexagonal lattice
                spacing = 10
                for i in range(0, img_size, spacing):
                    for j in range(0, img_size, int(spacing * 0.866)):
                        offset = spacing//2 if (j//int(spacing*0.866)) % 2 else 0
                        x, y = i + offset, j
                        if 0 <= x < img_size-2 and 0 <= y < img_size-2:
                            img[x:x+2, y:y+2] = 1.0
            
            elif class_id == 2:  # Random defects
                n_atoms = np.random.randint(15, 25)
                for _ in range(n_atoms):
                    x = np.random.randint(0, img_size-2)
                    y = np.random.randint(0, img_size-2)
                    img[x:x+2, y:y+2] = 1.0
            
            else:  # Diamond lattice pattern
                spacing = 12
                for i in range(0, img_size, spacing):
                    for j in range(0, img_size, spacing):
                        if i+2 < img_size and j+2 < img_size:
                            img[i:i+2, j:j+2] = 1.0
                        if i+spacing//2+2 < img_size and j+spacing//2+2 < img_size:
                            img[i+spacing//2:i+spacing//2+2, j+spacing//2:j+spacing//2+2] = 1.0
            
            # Apply Gaussian blur to simulate atomic columns
            img = ndimage.gaussian_filter(img, sigma=1.5)
            
            # Add noise
            noise = np.random.normal(0, 0.05, (img_size, img_size))
            img = img + noise
            
            # Normalize
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            
            images.append(img)
            labels.append(class_id)
    
    images = np.array(images)
    images = images.reshape(-1, img_size, img_size, 1)
    labels = np.array(labels)
    
    # Shuffle
    indices = np.random.permutation(len(images))
    images = images[indices]
    labels = labels[indices]
    
    return images, labels

# Generate dataset
print("Generating synthetic atomic patterns...")
X, y = generate_atomic_patterns(n_samples=400, img_size=64, n_classes=4)
print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Split into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"\nTrain set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

## Task 2: Build CNN Classifier

Build a convolutional neural network for pattern classification.

In [ ]:
def build_cnn_classifier(input_shape=(64, 64, 1), n_classes=4):
    """
    Build a CNN for atomic pattern classification.
    
    Args:
        input_shape: Shape of input images
        n_classes: Number of output classes
    
    Returns:
        model: Compiled Keras model
    """
    model = Sequential([
        # First convolutional block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second convolutional block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third convolutional block
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(n_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build model
model = build_cnn_classifier(input_shape=(64, 64, 1), n_classes=4)
model.summary()

## Task 3: Train the Model

Train the CNN with early stopping.

In [ ]:
# Set up callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train model
print("Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("\nTraining complete!")

## Task 4: Evaluate Model Performance

Evaluate the trained model on the test set.

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Calculate confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)
print("\nConfusion Matrix:")
print(conf_matrix)

# Classification report
class_names = ['Square', 'Hexagonal', 'Defects', 'Diamond']
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, target_names=class_names))

## Task 5: Create and Save Four-Panel Summary Figure

**This is the main deliverable:** Create a comprehensive four-panel summary figure showing:
1. Sample images from each class
2. Training history (loss and accuracy)
3. Confusion matrix
4. Sample predictions with confidence scores

In [ ]:
# Create four-panel summary figure
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.subplots_adjust(hspace=0.35, wspace=0.3)

class_names = ['Square', 'Hexagonal', 'Defects', 'Diamond']

# Panel 1: Sample images from each class (top-left)
ax1 = axes[0, 0]
# Create nested grid for samples
from matplotlib.gridspec import GridSpecFromSubplotSpec
gs1 = GridSpecFromSubplotSpec(2, 4, subplot_spec=ax1.get_subplotspec(), hspace=0.3, wspace=0.1)
for class_id in range(4):
    # Get two samples from each class
    class_indices = np.where(y_test == class_id)[0][:2]
    for i, idx in enumerate(class_indices):
        sub_ax = fig.add_subplot(gs1[i, class_id])
        sub_ax.imshow(X_test[idx].squeeze(), cmap='gray')
        if i == 0:
            sub_ax.set_title(class_names[class_id], fontsize=9, fontweight='bold')
        sub_ax.axis('off')
ax1.axis('off')
ax1.set_title('Panel 1: Sample Images from Each Class', fontsize=13, fontweight='bold', pad=15)

# Panel 2: Training history (top-right)
ax2 = axes[0, 1]
# Create nested grid for history plots
gs2 = GridSpecFromSubplotSpec(1, 2, subplot_spec=ax2.get_subplotspec(), wspace=0.3)
# Loss subplot
loss_ax = fig.add_subplot(gs2[0, 0])
loss_ax.plot(history.history['loss'], label='Training Loss', linewidth=2, color='#2E86AB')
loss_ax.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='#A23B72')
loss_ax.set_xlabel('Epoch', fontsize=10)
loss_ax.set_ylabel('Loss', fontsize=10)
loss_ax.set_title('Model Loss', fontsize=10, fontweight='bold')
loss_ax.legend(fontsize=8)
loss_ax.grid(True, alpha=0.3)
# Accuracy subplot
acc_ax = fig.add_subplot(gs2[0, 1])
acc_ax.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, color='#2E86AB')
acc_ax.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='#A23B72')
acc_ax.set_xlabel('Epoch', fontsize=10)
acc_ax.set_ylabel('Accuracy', fontsize=10)
acc_ax.set_title('Model Accuracy', fontsize=10, fontweight='bold')
acc_ax.legend(fontsize=8)
acc_ax.grid(True, alpha=0.3)
ax2.axis('off')
ax2.set_title('Panel 2: Training History', fontsize=13, fontweight='bold', pad=15)

# Panel 3: Confusion matrix (bottom-left)
ax3 = axes[1, 0]
# Create confusion matrix heatmap
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax3, square=True)
ax3.set_xlabel('Predicted Label', fontsize=11, fontweight='bold')
ax3.set_ylabel('True Label', fontsize=11, fontweight='bold')
ax3.set_title('Panel 3: Confusion Matrix', fontsize=13, fontweight='bold', pad=10)

# Panel 4: Sample predictions with confidence (bottom-right)
ax4 = axes[1, 1]
# Create nested grid for predictions
gs4 = GridSpecFromSubplotSpec(2, 4, subplot_spec=ax4.get_subplotspec(), hspace=0.4, wspace=0.1)
sample_indices = np.random.choice(len(X_test), size=8, replace=False)
for i, idx in enumerate(sample_indices):
    row, col = i // 4, i % 4
    sub_ax = fig.add_subplot(gs4[row, col])
    sub_ax.imshow(X_test[idx].squeeze(), cmap='gray')
    
    true_label = class_names[y_test[idx]]
    pred_label = class_names[y_pred_classes[idx]]
    confidence = y_pred[idx][y_pred_classes[idx]]
    
    color = 'green' if y_test[idx] == y_pred_classes[idx] else 'red'
    sub_ax.set_title(f'T: {true_label}\nP: {pred_label} ({confidence:.2f})', 
                     fontsize=7, color=color)
    sub_ax.axis('off')
ax4.axis('off')
ax4.set_title('Panel 4: Sample Predictions (Green=Correct, Red=Incorrect)', 
              fontsize=13, fontweight='bold', pad=15)

# Add overall title
fig.suptitle('Assignment 04: Four-Panel Summary - Atomic Pattern Recognition', 
             fontsize=16, fontweight='bold', y=0.98)

# Save the figure
output_dir = Path('figures')
output_dir.mkdir(exist_ok=True)
output_path = output_dir / 'assignment_04_summary_figure.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Four-panel summary figure saved to: {output_path}")
print(f"  Figure size: {fig.get_size_inches()[0]:.1f}\" x {fig.get_size_inches()[1]:.1f}\"")
print(f"  Resolution: 300 DPI")

plt.show()

print("\n" + "="*60)
print("ASSIGNMENT 04 COMPLETE")
print("="*60)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Summary figure saved: {output_path}")
print("="*60)


## Conclusion

This assignment demonstrated:
1. ✓ Generation of synthetic atomic pattern datasets
2. ✓ Building and training a CNN classifier
3. ✓ Model evaluation with multiple metrics
4. ✓ **Creation and saving of a comprehensive four-panel summary figure**

The four-panel summary provides a complete overview of the model's performance, including:
- Sample images from each class
- Training history (loss and accuracy curves)
- Confusion matrix for detailed performance analysis
- Sample predictions with confidence scores

This visualization approach is essential for communicating machine learning results in scientific publications and presentations.